# Clustering dos CPE com base nas features extraídas

o clustering é feito primeiro com o Set A (perfil horário normalizado - 24 features em %).

As features do Set A já estão na mesma escala (0-100%), pelo que NÃO requerem StandardScaler. Isto simplifica o pipeline e preserva a interpretabilidade directa dos valores.

Estrutura:
- Carregamento do Set A
- Análise prévia (outliers, PCA para visualização)
- Seleção do K (4 métricas)
- KMeans - análise detalhada (orientação do supervisor)
- Comparação com outros métodos
- Interpretação dos clusters (perfis horários)
- Exportação


In [ ]:
# Imports e configuração

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
 
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
 
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")

base_dir = Path("../results")

# Pastas principais
exploration_dir = base_dir / "exploration"
features_dir = base_dir / "features"
clustering_dir = base_dir / "clustering"
anomalies_dir = base_dir / "anomalies"
models_dir = base_dir / "models"

# Subpastas específicas
intermediate_dir = exploration_dir / "intermediate"

# Clustering
clustering_analysis_dir = clustering_dir / "analysis"
clustering_plots_dir = clustering_dir / "plots"

# Features
features_plots_dir = features_dir / "plots"
features_validation_dir = features_dir / "validation"

# Anomalies
anomalies_plots_dir = anomalies_dir / "plots"
anomalies_temporal_plots_dir = anomalies_dir / "anomalies_temporal_plots"

for d in [
    exploration_dir,
    features_dir,
    clustering_dir,
    anomalies_dir,
    models_dir,
    intermediate_dir,
    clustering_analysis_dir,
    clustering_plots_dir,
    features_plots_dir,
    features_validation_dir,
    anomalies_plots_dir,
    anomalies_temporal_plots_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# Carregamento do Set A

features = pd.read_csv(features_dir / "features_setA.csv", index_col=0)
 
print("Shape:", features.shape)
print("Número de CPE:", features.index.nunique())
print("Número de features:", features.shape[1])
print("\nPrimeiros CPEs:")
display(features.head())
 
# Verificação: as features somam ~100% por CPE
somas = features.sum(axis=1)
print(f"\nSoma por CPE — Min: {somas.min():.1f}%, Max: {somas.max():.1f}%, Mean: {somas.mean():.1f}%")

## Observações

O Set A contém 24 features homogéneas (% de consumo por hora). Todas estão na mesma escala, pelo que NÃO é necessário aplicar StandardScaler - fazer clustering diretamente sobre os dados originais preserva a interpretabilidade.

In [ ]:
# Preparação dos dados

X = features.copy()
feature_names = X.columns.tolist()
 
# Como as features já estão na mesma escala, usamos os dados directamente
# para clustering. O StandardScaler NÃO é aplicado.
X_values = X.values
 
# PCA para visualização (apenas para projectar em 2D, não para clustering)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_values)
 
df_pca = pd.DataFrame(X_pca, index=X.index, columns=["PC1", "PC2"])
 
explained_variance = pca.explained_variance_ratio_
print(f"Variância explicada (PCA):")
print(f"  PC1: {explained_variance[0]:.2%}")
print(f"  PC2: {explained_variance[1]:.2%}")
print(f"  Total: {explained_variance.sum():.2%}")
 
# Persistir PCA
joblib.dump(pca, models_dir / "pca.pkl")
print(f"PCA guardado: {models_dir / 'pca.pkl'}")
 
# Visualização inicial
plt.figure(figsize=(8, 6))
plt.scatter(df_pca["PC1"], df_pca["PC2"], alpha=0.7)
plt.title("Projeção dos CPE no espaço PCA")
plt.xlabel(f"PC1 ({explained_variance[0]:.1%} variância)")
plt.ylabel(f"PC2 ({explained_variance[1]:.1%} variância)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Deteção prévia de outliers

# Mesmo com features homogéneas, pode haver CPEs com perfis extremamente
# atípicos que distorçam o clustering.
 
contam_values = [0.01, 0.02, 0.03, 0.05]
for contam in contam_values:
    iso = IsolationForest(contamination=contam, random_state=42)
    labels = iso.fit_predict(X_values)
    n_out = (labels == -1).sum()
    print(f"  contamination={contam:.2f} → {n_out} outliers ({n_out/len(labels)*100:.1f}%)")
 
CONTAM_ESCOLHIDA = 0.02
iso = IsolationForest(contamination=CONTAM_ESCOLHIDA, random_state=42)
outlier_labels = iso.fit_predict(X_values)
 
mask_normal = outlier_labels == 1
X_clean = X.loc[mask_normal].copy()
X_clean_values = X_values[mask_normal]
X_outliers = X.loc[~mask_normal].copy()
 
print(f"\nContamination escolhida: {CONTAM_ESCOLHIDA}")
print(f"Shape original: {X.shape} → Shape sem outliers: {X_clean.shape}")
print(f"CPEs outliers: {X_outliers.index.tolist()}")
 
# Atualizar PCA sem outliers
X_pca_clean = pca.transform(X_clean_values)
df_pca = pd.DataFrame(X_pca_clean, index=X_clean.index, columns=["PC1", "PC2"])
 
# Guardar mask
pd.Series(mask_normal, index=X.index, name="is_normal").to_csv(clustering_dir / "outlier_mask.csv")

In [ ]:
# Seleção do K

K_range = range(2, 11)
inertias = []
silhouette_list = []
calinski_list = []
davies_list = []
 
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clean_values)
    
    inertias.append(km.inertia_)
    silhouette_list.append(silhouette_score(X_clean_values, labels))
    calinski_list.append(calinski_harabasz_score(X_clean_values, labels))
    davies_list.append(davies_bouldin_score(X_clean_values, labels))
 
df_k = pd.DataFrame({
    "K": list(K_range),
    "Inertia": inertias,
    "Silhouette": silhouette_list,
    "Calinski-Harabasz": calinski_list,
    "Davies-Bouldin": davies_list
})
display(df_k)
 
# Gráficos
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
 
axes[0,0].plot(K_range, inertias, marker="o", color="#0D7C66")
axes[0,0].set_title("Método do Cotovelo (Elbow)")
axes[0,0].set_xlabel("K"); axes[0,0].set_ylabel("Inércia")
axes[0,0].set_xticks(list(K_range))
 
axes[0,1].plot(K_range, silhouette_list, marker="o", color="#14A38B")
axes[0,1].set_title("Silhouette Score")
axes[0,1].set_xlabel("K"); axes[0,1].set_ylabel("Score")
axes[0,1].set_xticks(list(K_range))
 
axes[1,0].plot(K_range, calinski_list, marker="o", color="#2980B9")
axes[1,0].set_title("Calinski-Harabasz (↑ melhor)")
axes[1,0].set_xlabel("K"); axes[1,0].set_ylabel("Score")
axes[1,0].set_xticks(list(K_range))
 
axes[1,1].plot(K_range, davies_list, marker="o", color="#E74C3C")
axes[1,1].set_title("Davies-Bouldin (↓ melhor)")
axes[1,1].set_xlabel("K"); axes[1,1].set_ylabel("Score")
axes[1,1].set_xticks(list(K_range))
 
plt.suptitle("Seleção do número de clusters (Set A)", fontweight="bold")
plt.tight_layout()
plt.savefig(clustering_analysis_dir / "selecao_k_setA.png")
plt.show()
 
# Melhor K por métrica
best_sil = df_k.loc[df_k["Silhouette"].idxmax(), "K"]
best_cal = df_k.loc[df_k["Calinski-Harabasz"].idxmax(), "K"]
best_dav = df_k.loc[df_k["Davies-Bouldin"].idxmin(), "K"]
print(f"Melhor K — Silhouette: {best_sil}, Calinski: {best_cal}, Davies: {best_dav}")
 
K_opt = 3  # Escolha fundamentada na análise
print(f"\nK escolhido: {K_opt}")

In [ ]:
# KMeans — análise detalhada

kmeans = KMeans(n_clusters=K_opt, random_state=42, n_init=10)
clusters_labels = kmeans.fit_predict(X_clean_values)
 
df_clusters = X_clean.copy()
df_clusters["cluster"] = clusters_labels
 
print(f"Número de CPE por cluster:")
display(df_clusters["cluster"].value_counts().sort_index())
 
# Persistir modelo
joblib.dump(kmeans, models_dir / "kmeans.pkl")
 
# --- Visualização PCA ---
df_pca["cluster"] = clusters_labels
 
fig, ax = plt.subplots(figsize=(8, 6))
cores = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
         "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
 
for cid in sorted(df_pca["cluster"].unique()):
    mask = df_pca["cluster"] == cid
    ax.scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC2"],
               c=cores[cid], label=f"Cluster {cid}", alpha=0.7, s=50)
 
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           c="black", s=200, marker="X", label="Centróides", zorder=5)
 
ax.set_title(f"Clusters de consumo dos CPE (KMeans K={K_opt}, projeção PCA)")
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1%} variância)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1%} variância)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(clustering_plots_dir / "clusters_pca_setA.png")
plt.show()
 
# --- Silhouette por CPE ---
sample_sil = silhouette_samples(X_clean_values, clusters_labels)
 
fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10
for i in sorted(set(clusters_labels)):
    vals = sample_sil[clusters_labels == i]
    vals.sort()
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * len(vals), str(i))
    y_lower = y_upper + 10
 
ax.axvline(x=silhouette_score(X_clean_values, clusters_labels),
           color="red", linestyle="--", label="Média")
ax.set_title("Silhouette por CPE e cluster (Set A)")
ax.set_xlabel("Silhouette Score")
ax.set_ylabel("Cluster")
ax.legend()
plt.tight_layout()
plt.savefig(clustering_analysis_dir / "silhouette_setA.png", dpi=150)
plt.show()

In [ ]:
# Interpretação dos clusters — perfil horário

# Com o Set A, a interpretação é directa: cada cluster tem um perfil
# horário médio que podemos visualizar.
 
print("INTERPRETAÇÃO DOS CLUSTERS — PERFIL HORÁRIO")
 
cluster_summary = df_clusters.groupby("cluster").mean()
 
# Perfil horário por cluster
horas = list(range(24))
fig, ax = plt.subplots(figsize=(12, 5))
 
for cid in sorted(cluster_summary.index):
    n_cpes = (df_clusters["cluster"] == cid).sum()
    perfil = cluster_summary.loc[cid].values
    ax.plot(horas, perfil, marker="o", markersize=4,
            label=f"Cluster {cid} ({n_cpes} CPEs)", linewidth=2)
 
ax.set_title("Perfil horário médio por cluster (% consumo diário)")
ax.set_xlabel("Hora do dia")
ax.set_ylabel("% do consumo diário")
ax.set_xticks(horas)
ax.set_xticklabels([f"{h}h" for h in horas], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(clustering_plots_dir / "perfil_horario_por_cluster.png", dpi=150)
plt.show()
 
# Heatmap dos perfis por cluster
plt.figure(figsize=(14, 4))
sns.heatmap(
    cluster_summary,
    annot=True, fmt=".1f", cmap="YlOrRd",
    xticklabels=[f"{h}h" for h in horas],
    yticklabels=[f"Cluster {i}" for i in cluster_summary.index],
    cbar_kws={"label": "% consumo diário"}
)
plt.title("Perfil horário por cluster (% consumo diário)")
plt.xlabel("Hora do dia")
plt.tight_layout()
plt.savefig(clustering_plots_dir / "heatmap_clusters_setA.png", dpi=150)
plt.show()
 
# Interpretação automática
for cid in sorted(cluster_summary.index):
    n_cpes = (df_clusters["cluster"] == cid).sum()
    perfil = cluster_summary.loc[cid]
    hora_pico = perfil.idxmax()
    hora_num = int(hora_pico.split("_")[-1])
    pct_noite = perfil.iloc[0:7].sum()  # 0h-6h
    pct_dia = perfil.iloc[7:21].sum()    # 7h-20h
    pct_tarde_noite = perfil.iloc[21:24].sum()  # 21h-23h
    
    print(f"\nCluster {cid} ({n_cpes} CPEs)")
    print(f"  Hora de pico: {hora_num}h ({perfil.max():.1f}%)")
    print(f"  Consumo noturno (0-6h): {pct_noite:.1f}%")
    print(f"  Consumo diurno (7-20h): {pct_dia:.1f}%")
    print(f"  Consumo tardio (21-23h): {pct_tarde_noite:.1f}%")
    
    if pct_dia > 70:
        print(f"  → Perfil predominantemente DIURNO")
    elif pct_noite > 40:
        print(f"  → Perfil com forte componente NOTURNA")
    else:
        print(f"  → Perfil MISTO / equilibrado")

In [ ]:
# Comparação de métodos

print("COMPARAÇÃO DE MÉTODOS DE CLUSTERING")
 
resultados_metodos = {}
 
# KMeans (já calculado)
resultados_metodos["KMeans"] = {
    "labels": clusters_labels,
    "silhouette": silhouette_score(X_clean_values, clusters_labels),
    "calinski": calinski_harabasz_score(X_clean_values, clusters_labels),
    "davies": davies_bouldin_score(X_clean_values, clusters_labels),
    "n_clusters": K_opt,
}
 
# Agglomerative
labels_agglo = AgglomerativeClustering(n_clusters=K_opt).fit_predict(X_clean_values)
resultados_metodos["Agglomerative"] = {
    "labels": labels_agglo,
    "silhouette": silhouette_score(X_clean_values, labels_agglo),
    "calinski": calinski_harabasz_score(X_clean_values, labels_agglo),
    "davies": davies_bouldin_score(X_clean_values, labels_agglo),
    "n_clusters": K_opt,
}
 
# GMM
labels_gmm = GaussianMixture(n_components=K_opt, random_state=42).fit_predict(X_clean_values)
resultados_metodos["GMM"] = {
    "labels": labels_gmm,
    "silhouette": silhouette_score(X_clean_values, labels_gmm),
    "calinski": calinski_harabasz_score(X_clean_values, labels_gmm),
    "davies": davies_bouldin_score(X_clean_values, labels_gmm),
    "n_clusters": K_opt,
}
 
# DBSCAN
nn = NearestNeighbors(n_neighbors=5).fit(X_clean_values)
distances, _ = nn.kneighbors(X_clean_values)
k_distances = np.sort(distances[:, -1])
 
plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.title("K-distance plot (k=5) para estimação de eps")
plt.xlabel("Pontos ordenados"); plt.ylabel("Distância ao 5º vizinho")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
 
for eps in [2.0, 3.0, 4.0, 5.0, 6.0, 8.0]:
    db = DBSCAN(eps=eps, min_samples=3).fit_predict(X_clean_values)
    nc = len(set(db)) - (1 if -1 in db else 0)
    nn_noise = (db == -1).sum()
    sil = silhouette_score(X_clean_values, db) if nc >= 2 else -1
    print(f"  eps={eps:.1f} → {nc} clusters, {nn_noise} noise, silhouette={sil:.3f}")
 
# Tabela comparativa
df_comp = pd.DataFrame({
    m: {"Silhouette ↑": round(r["silhouette"], 3),
        "Calinski ↑": round(r["calinski"], 1),
        "Davies ↓": round(r["davies"], 3)}
    for m, r in resultados_metodos.items()
}).T
print("\nComparação de métodos:")
display(df_comp)
df_comp.to_csv(clustering_dir / "comparacao_clustering_setA.csv")
 
# Visualização PCA por método
fig, axes = plt.subplots(1, len(resultados_metodos), figsize=(5*len(resultados_metodos), 5))
for ax, (method, res) in zip(axes, resultados_metodos.items()):
    for cid in sorted(set(res["labels"])):
        mask = res["labels"] == cid
        ax.scatter(df_pca.loc[mask, "PC1"], df_pca.loc[mask, "PC2"],
                   c=cores[cid], alpha=0.7, s=40, label=f"C{cid}")
    ax.set_title(f"{method}\nSil: {res['silhouette']:.3f}")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.3)
 
plt.suptitle("Comparação de métodos (Set A)", fontweight="bold")
plt.tight_layout()
plt.savefig(clustering_analysis_dir / "comparacao_metodos_pca_setA.png", dpi=150)
plt.show()

In [ ]:
# Exportação

# Juntar outliers
df_outlier_result = X_outliers.copy()
df_outlier_result["cluster"] = "outlier"
 
df_clusters_final = pd.concat([df_clusters, df_outlier_result])
 
print(f"\nDistribuição final:")
display(df_clusters_final["cluster"].value_counts())
 
df_clusters_final.to_csv(clustering_dir / "clusters_cpe.csv")
df_pca.to_csv(clustering_dir / "pca_clusters_cpe.csv")
cluster_summary.to_csv(clustering_dir / "cluster_summary.csv")
 
print("\nFicheiros guardados:")
print(f"  {clustering_dir / 'clusters_cpe.csv'}")
print(f"  {clustering_dir / 'pca_clusters_cpe.csv'}")
print(f"  {clustering_dir / 'cluster_summary.csv'}")
 
print(f"\nModelos persistidos:")
print(f"  {models_dir / 'pca.pkl'}")
print(f"  {models_dir / 'kmeans.pkl'}")

# Conclusões

O clustering com Set A (perfil horário normalizado) permite identificar tipologias de consumo baseadas exclusivamente no PADRÃO temporal de utilização, independentemente da intensidade absoluta do consumo.

Esta abordagem é mais interpretável: cada cluster corresponde a um perfil horário distinto (diurno, noturno, misto, etc.).

Próximo passo (orientação do supervisor):
- Repetir com Set B e comparar resultados
- Avaliar qual conjunto de features produz clusters mais úteis do ponto de vista dos utilizadores